(Positivity-ADR)=
# Positivity and mass preserving ADR

Given the application we are interested in, i.e. adr equations describing chemical reactions, it is of interest to have a way to preserve positivity. This is due to the fact that negative concentrations of some species are not physical and lead to faulty behavior.

To exemplify the phenomena let's consider a prototype cell (circle) in which some chemical is pushed to the edge by internal forces. We will force, using the keyword `Fneu_b`, zero flux at the boundary. This leads to an ill-defined problem mathematically, since we will use $\vec{b}=(0.1,0)$. Such a velocity have a non-zero trace on the boundary and is directed outward with respect to the surface normal. This should let the chemical flow out of the cell, but the forced boundary conditions do not let it pass and this generates oscillation. In practice it is there was a very steep boundary layer at the edge where the velocity suddently drops to zero.

Consider thus the linear advection equation

\begin{equation*}
\partial_t u + \nabla\cdot(\vec{b}u - d \nabla u)= 0
\end{equation*}

with the ususal terminology as in [the bulk-ADR example](Bulk-Adr). Imposing zero flux should lead to a mass preserving behaviour.

Let's tart by creating our prototype "cell".

In [ ]:
from cosmos.utils.generate_surface_meshes import generate_circle
from ngsolve.webgui import Draw

mesh, _ = generate_circle()
Draw(mesh)


Now let's initialize the problem as ususal

In [ ]:
from ngsolve import *
from cosmos.solvers.simulation import Simulation

dt = Parameter(0.1)
T = 10
t = Parameter(0)
simulation = Simulation(mesh=mesh, t=t, dt = dt, T = T)

Define the solver and the coefficients. We begin with a Gaussian distribution at the center of the cell.

In [ ]:
from cosmos.solvers.adr import ADRSolver

solver = ADRSolver(linear = True)
simulation.AddSolver(solver)

from cosmos.solvers.tools import gradient
from ngsolve.webgui import Draw

u0 = exp(-4*(x**2+y**2))
Draw(u0, mesh)
b = CF((0.1,0))
d = 0.0
Fflux = CF((0,0))
Fneu_b = {'boundary': Fflux}

Subsequently we add the solver to the simulation



In [ ]:
solver.AddSpecie(VorB = VOL,
                advection = b,
                diffusion = d,
                u0 = u0,
                Fneu_b = Fneu_b)

We can finally run the simulation and plot the final result, to see what happened.

In [ ]:
simulation.run()
print('Computed solution')
solver.draw_solution()


We can verify that the mass is indeed conserved (see below, up to numerical integration in time errors), but it is clear that the concentration of the chemical has become unstable leading to negative concentrations

In [ ]:
mass0 = Integrate(u0, mesh, order = 3)
mass_end = Integrate(solver.get_solution(), mesh, order = 3)

print('Relative mass error: ', (mass_end-mass0)/mass0)

We can fix this by setting the keyword `PP` to `True`, so that the algorithm takes care to preserve positivity. If we also need mass preserving, as in this case, we set instead `MP` to `True`.

In [ ]:
solver_mp = ADRSolver(linear = True)
simulation.AddSolver(solver_mp)

solver_mp.AddSpecie(VorB = VOL,
                advection = b,
                diffusion = d,
                u0 = u0,
                Fneu_b = Fneu_b,
                MP = True)

t.Set(0.0)
simulation.run()
print('Computed solution')
solver_mp.draw_solution()

mass_end_mp = Integrate(solver_mp.get_solution(), mesh, order = 3)

print('Relative mass error: ', (mass_end_mp-mass0)/mass0)

The algorithm used to obtain so can be found [here](https://www.sciencedirect.com/science/article/pii/S0045782522000093?ref=pdf_download&fr=RR-2&rr=914d9bcd4f4292c1) and following.